# Curate Caldova launch-readiness tables

This notebook downloads the six Caldova source files to `Files/caldova-data`, applies explicit data types, writes managed Delta tables in the `dbo` schema, validates row counts and relationships, and displays the LP001 decision evidence.

In [ ]:
from urllib.request import urlopen
from pyspark.sql import functions as F

SOURCE_ROOT = "https://raw.githubusercontent.com/MicrosoftLearning/mslearn-unify-data/main/Allfiles/L300-unify-data/data"
SOURCE_FILES = [
    "launch_program.csv",
    "manufacturing_batch_schedule.csv",
    "manufacturing_capacity_plan.csv",
    "plants.csv",
    "procurement_cmo_evaluation.csv",
    "products.csv",
]
DATA_ROOT = "/lakehouse/default/Files/caldova-data"

notebookutils.fs.mkdirs("Files/caldova-data")
for file_name in SOURCE_FILES:
    with urlopen(f"{SOURCE_ROOT}/{file_name}", timeout=60) as response:
        content = response.read().decode("utf-8-sig")
    notebookutils.fs.put(f"Files/caldova-data/{file_name}", content, overwrite=True)
    print(f"Downloaded {file_name}")

def read_csv(file_name, schema):
    return (spark.read
            .option("header", True)
            .option("encoding", "UTF-8")
            .schema(schema)
            .csv(f"{DATA_ROOT}/{file_name}"))

def write_table(dataframe, table_name):
    (dataframe.write
     .format("delta")
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable(f"dbo.{table_name}"))
    print(f"Wrote {table_name}: {dataframe.count()} rows")

In [ ]:
launch_program = read_csv(
    "launch_program.csv",
    """LaunchProgramID string, LaunchProgramName string, ProductID string,
       TargetLaunchDate string, TargetLaunchVolume long, QualifiedCapacity long,
       CapacityGapUnits long, CapacityGapPercent double,
       BindingConstraintPlantID string, Status string"""
).withColumn(
    "TargetLaunchDate",
    F.to_date(F.substring("TargetLaunchDate", 1, 10), "yyyy-MM-dd"),
) \
 .withColumn(
     "BindingConstraintPlantID",
     F.when(F.trim("BindingConstraintPlantID") == "", None)
      .otherwise(F.col("BindingConstraintPlantID")),
 )

product = read_csv(
    "products.csv",
    """ProductID string, ProductName string, TherapyArea string,
       LifecycleStage string, CommercialRegion string, ManufacturingType string"""
)

plant = read_csv(
    "plants.csv",
    """PlantID string, PlantName string, Country string, Region string,
       PlantType string, GMPStatus string, PlantManager string"""
)

write_table(launch_program, "launch_program")
write_table(product, "product")
write_table(plant, "plant")

In [ ]:
capacity_plan = read_csv(
    "manufacturing_capacity_plan.csv",
    """CapacityPlanID string, LaunchProgramID string, PlantID string,
       ReportingMonth string, MaxCapacity long, CommittedCapacity long,
       AvailableCapacity long, UtilizationPercent double"""
).withColumn(
    "ReportingMonth",
    F.to_date(F.to_timestamp("ReportingMonth", "M/d/yyyy H:mm")),
)

batch = read_csv(
    "manufacturing_batch_schedule.csv",
    """BatchID string, LaunchProgramID string, PlantID string, ProductID string,
       BatchStatus string, ScheduledStartDate string, ScheduledEndDate string,
       ProductionVolume long, Priority string"""
).withColumn(
    "ScheduledStartDate", F.to_date("ScheduledStartDate", "M/d/yyyy")
).withColumn(
    "ScheduledEndDate", F.to_date("ScheduledEndDate", "M/d/yyyy")
)

supplier = read_csv(
    "procurement_cmo_evaluation.csv",
    """CMOID string, CMOName string, LaunchProgramID string, ProductID string,
       DosageForm string, GMPCertStatus string, LastAuditDate string,
       OpenCriticalFindings int, QualificationTier string,
       QualifiedAvailableCapacityUnits long, LeadTimeWeeks int,
       EstimatedValidationMonths int, FastTrackEligible string, TechnicalFit string,
       CommercialRiskTier string, GMPQualityScore int, CapacityScore int,
       FastTrackScore int, TechnicalFitScore int, CommercialScore int,
       WeightedScore int, Recommendation string, RFPStatus string"""
).withColumn(
    "LastAuditDate",
    F.to_date(
        F.regexp_replace(
            F.regexp_replace("LastAuditDate", r"(?i)(?<=\\d)(st|nd|rd|th)", ""),
            r"(?i)\\bSept\\b",
            "Sep",
        ),
        "d MMM yyyy",
    ),
)

write_table(capacity_plan, "capacity_plan")
write_table(batch, "batch")
write_table(supplier, "supplier")

In [ ]:
expected_counts = {
    "launch_program": 10,
    "product": 25,
    "plant": 5,
    "capacity_plan": 60,
    "batch": 212,
    "supplier": 6,
}

for table_name, expected in expected_counts.items():
    actual = spark.table(f"dbo.{table_name}").count()
    if actual != expected:
        raise AssertionError(f"{table_name}: expected {expected}, found {actual}")

relationship_checks = {
    "launch_program.ProductID -> product": launch_program.join(product, "ProductID", "left_anti").count(),
    "launch_program.BindingConstraintPlantID -> plant": (
        launch_program.where(F.col("BindingConstraintPlantID").isNotNull())
        .join(plant, launch_program.BindingConstraintPlantID == plant.PlantID, "left_anti")
        .count()
    ),
    "capacity_plan.LaunchProgramID -> launch_program": capacity_plan.join(launch_program, "LaunchProgramID", "left_anti").count(),
    "capacity_plan.PlantID -> plant": capacity_plan.join(plant, "PlantID", "left_anti").count(),
    "batch.LaunchProgramID -> launch_program": batch.join(launch_program, "LaunchProgramID", "left_anti").count(),
    "batch.PlantID -> plant": batch.join(plant, "PlantID", "left_anti").count(),
    "batch.ProductID -> product": batch.join(product, "ProductID", "left_anti").count(),
    "supplier.LaunchProgramID -> launch_program": supplier.join(launch_program, "LaunchProgramID", "left_anti").count(),
    "supplier.ProductID -> product": supplier.join(product, "ProductID", "left_anti").count(),
}

failed = {name: count for name, count in relationship_checks.items() if count != 0}
if failed:
    raise AssertionError(f"Orphaned relationship keys: {failed}")

print("Validation passed: 318 rows across six tables; all tested relationships are valid.")

In [ ]:
display(spark.sql("""
WITH delayed AS (
  SELECT LaunchProgramID, COUNT(*) AS DelayedBatchCount
  FROM dbo.batch
  WHERE BatchStatus = 'Delayed'
  GROUP BY LaunchProgramID
)
SELECT lp.LaunchProgramID, lp.LaunchProgramName, p.ProductName,
       lp.TargetLaunchVolume, lp.QualifiedCapacity, lp.CapacityGapUnits,
       lp.CapacityGapPercent, pl.PlantName AS BindingPlant,
       COALESCE(d.DelayedBatchCount, 0) AS DelayedBatchCount, lp.Status
FROM launch_program lp
JOIN dbo.product p ON lp.ProductID = p.ProductID
LEFT JOIN dbo.plant pl ON lp.BindingConstraintPlantID = pl.PlantID
LEFT JOIN delayed d ON lp.LaunchProgramID = d.LaunchProgramID
WHERE lp.LaunchProgramID = 'LP001'
"""))

display(spark.sql("""
SELECT s.CMOID, s.CMOName, s.QualifiedAvailableCapacityUnits,
       s.LeadTimeWeeks, s.GMPCertStatus, s.OpenCriticalFindings,
       s.GMPQualityScore, s.WeightedScore, s.Recommendation
FROM supplier s
JOIN dbo.launch_program lp ON s.LaunchProgramID = lp.LaunchProgramID
WHERE s.LaunchProgramID = 'LP001'
  AND s.QualifiedAvailableCapacityUnits >= lp.CapacityGapUnits
  AND s.GMPCertStatus = 'Currently valid'
  AND s.OpenCriticalFindings = 0
  AND s.FastTrackEligible = 'Yes'
ORDER BY s.WeightedScore DESC, s.LeadTimeWeeks ASC
"""))